# DMEPOS Amount stats exploration

In this workbook we engineer a new set of features calculating mean, median, minimum, maximum, and standard deviation of numeric data fields for each unique NPI aggregated by program year. We also engineer an additional two features, `Tot_Suplr_Nonrntl_HCPCS_Cds` and `Tot_Suplr_Rentl_HCPCS_Cds` corresponding to the count of unique HCPCS codes with the negative and positive rental tag respectively for each unique NPI aggregated by the program year.

# Aggregation needs to be done at the NPI-Year Level

# Narsim's data is aggregated at NPI-level only. Retain work as Raw NB convert cells for documentation only.

# Data Aggregation/Feature Engineering

In [1]:
import pandas as pd

pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)

# read-in the dataset for validation
df_validate = pd.read_csv('/dsa/groups/casestudycf25/team02/DMEPOS_rfrhpr_clean_labeled.csv',dtype={'Rfrg_Prvdr_State_FIPS':str,'Rfrg_Prvdr_Zip5':str})
df_validate.head()

,npi,Rfrg_Prvdr_Last_Name_Org,Rfrg_Prvdr_First_Name,Rfrg_Prvdr_MI,Rfrg_Prvdr_Crdntls,Rfrg_Prvdr_Ent_Cd,Rfrg_Prvdr_St1,Rfrg_Prvdr_St2,Rfrg_Prvdr_City,Rfrg_Prvdr_State_Abrvtn,Rfrg_Prvdr_State_FIPS,Rfrg_Prvdr_Zip5,Rfrg_Prvdr_RUCA_Cat,Rfrg_Prvdr_RUCA,Rfrg_Prvdr_RUCA_Desc,Rfrg_Prvdr_Cntry,Rfrg_Prvdr_Spclty_Cd,Rfrg_Prvdr_Spclty_Desc,Rfrg_Prvdr_Spclty_Srce,RBCS_Lvl,RBCS_Id,RBCS_Desc,HCPCS_CD,HCPCS_Desc,Suplr_Rentl_Ind,Tot_Suplrs,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Avg_Suplr_Sbmtd_Chrg,Avg_Suplr_Mdcr_Alowd_Amt,Avg_Suplr_Mdcr_Pymt_Amt,Avg_Suplr_Mdcr_Stdzd_Amt,Year,target
0,1265697478,Balger,Abigail,NaN,md,I,805 E Geneva Dr,NaN,Dewitt,MI,26,48820,Urban,1.0,Metropolitan area core: primary flow within an...,US,93,Emergency Medicine,NPPES-Specialty,Orthotic Devices,DF007N,DME-Orthotic Devices,L0648,"Lumbar-sacral orthosis, sagittal control, with...",N,1,25.0,25,25,1100.000000,802.673200,623.734400,617.563600,2021,1
1,1265697478,Balger,Abigail,NaN,md,I,805 E Geneva Dr,NaN,Dewitt,MI,26,48820,Urban,1.0,Metropolitan area core: primary flow within an...,US,93,Emergency Medicine,NPPES-Specialty,Orthotic Devices,DF011N,DME-Orthotic Devices,L1833,"Knee orthosis, adjustable knee joints (unicent...",N,1,24.0,24,45,800.000000,575.352000,429.419333,388.558889,2021,1
2,1922096221,Alperovich,Alexander,NaN,md,I,1340 Union University Dr,NaN,Jackson,TN,47,38305,Urban,1.0,Metropolitan area core: primary flow within an...,US,06,Cardiology,Claim-Specialty,Durable Medical Equipment,DE001N,DME-Other DME,A7030,Full face mask used with positive airway press...,N,4,5.0,11,11,391.454545,127.299091,101.008182,88.409091,2021,1
3,1922096221,Alperovich,Alexander,NaN,md,I,1340 Union University Dr,NaN,Jackson,TN,47,38305,Urban,1.0,Metropolitan area core: primary flow within an...,US,06,Cardiology,Claim-Specialty,Durable Medical Equipment,DE001N,DME-Other DME,A7035,Headgear used with positive airway pressure de...,N,5,14.0,16,16,75.225000,26.028125,20.823750,18.594375,2021,1
4,1922096221,Alperovich,Alexander,NaN,md,I,1340 Union University Dr,NaN,Jackson,TN,47,38305,Urban,1.0,Metropolitan area core: primary flow within an...,US,06,Cardiology,Claim-Specialty,Durable Medical Equipment,DE001N,DME-Other DME,A7037,Tubing used with positive airway pressure device,N,5,12.0,16,16,80.843750,24.173750,18.241875,14.060000,2021,1


In [2]:
df_validate_eng = df_validate[['npi','Avg_Suplr_Sbmtd_Chrg','Avg_Suplr_Mdcr_Alowd_Amt','Avg_Suplr_Mdcr_Pymt_Amt','Avg_Suplr_Mdcr_Stdzd_Amt','Year','target']].groupby(['npi','Year','target']).mean().reset_index()
df_validate_eng = df_validate_eng.rename(columns={'Avg_Suplr_Sbmtd_Chrg':'Avg_Suplr_Sbmtd_Chrg_mean','Avg_Suplr_Mdcr_Alowd_Amt':'Avg_Suplr_Mdcr_Alowd_Amt_mean','Avg_Suplr_Mdcr_Pymt_Amt':'Avg_Suplr_Mdcr_Pymt_Amt_mean','Avg_Suplr_Mdcr_Stdzd_Amt':'Avg_Suplr_Mdcr_Stdzd_Amt_mean'})
df_validate_eng[['Avg_Suplr_Sbmtd_Chrg_sum','Avg_Suplr_Mdcr_Alowd_Amt_sum','Avg_Suplr_Mdcr_Pymt_Amt_sum','Avg_Suplr_Mdcr_Stdzd_Amt_sum']] = df_validate[['npi','Avg_Suplr_Sbmtd_Chrg','Avg_Suplr_Mdcr_Alowd_Amt','Avg_Suplr_Mdcr_Pymt_Amt','Avg_Suplr_Mdcr_Stdzd_Amt','Year','target']].groupby(['npi','Year','target']).sum().reset_index(drop=True)
df_validate_eng[['Avg_Suplr_Sbmtd_Chrg_median','Avg_Suplr_Mdcr_Alowd_Amt_median','Avg_Suplr_Mdcr_Pymt_Amt_median','Avg_Suplr_Mdcr_Stdzd_Amt_median']] = df_validate[['npi','Avg_Suplr_Sbmtd_Chrg','Avg_Suplr_Mdcr_Alowd_Amt','Avg_Suplr_Mdcr_Pymt_Amt','Avg_Suplr_Mdcr_Stdzd_Amt','Year','target']].groupby(['npi','Year','target']).median().reset_index(drop=True)
df_validate_eng[['Avg_Suplr_Sbmtd_Chrg_std','Avg_Suplr_Mdcr_Alowd_Amt_std','Avg_Suplr_Mdcr_Pymt_Amt_std','Avg_Suplr_Mdcr_Stdzd_Amt_std']] = df_validate[['npi','Avg_Suplr_Sbmtd_Chrg','Avg_Suplr_Mdcr_Alowd_Amt','Avg_Suplr_Mdcr_Pymt_Amt','Avg_Suplr_Mdcr_Stdzd_Amt','Year','target']].groupby(['npi','Year','target']).std().reset_index(drop=True)
df_validate_eng[['Avg_Suplr_Sbmtd_Chrg_min','Avg_Suplr_Mdcr_Alowd_Amt_min','Avg_Suplr_Mdcr_Pymt_Amt_min','Avg_Suplr_Mdcr_Stdzd_Amt_min']] = df_validate[['npi','Avg_Suplr_Sbmtd_Chrg','Avg_Suplr_Mdcr_Alowd_Amt','Avg_Suplr_Mdcr_Pymt_Amt','Avg_Suplr_Mdcr_Stdzd_Amt','Year','target']].groupby(['npi','Year','target']).min().reset_index(drop=True)
df_validate_eng[['Avg_Suplr_Sbmtd_Chrg_max','Avg_Suplr_Mdcr_Alowd_Amt_max','Avg_Suplr_Mdcr_Pymt_Amt_max','Avg_Suplr_Mdcr_Stdzd_Amt_max']] = df_validate[['npi','Avg_Suplr_Sbmtd_Chrg','Avg_Suplr_Mdcr_Alowd_Amt','Avg_Suplr_Mdcr_Pymt_Amt','Avg_Suplr_Mdcr_Stdzd_Amt','Year','target']].groupby(['npi','Year','target']).max().reset_index(drop=True)
df_validate_eng.head()

,npi,Year,target,Avg_Suplr_Sbmtd_Chrg_mean,Avg_Suplr_Mdcr_Alowd_Amt_mean,Avg_Suplr_Mdcr_Pymt_Amt_mean,Avg_Suplr_Mdcr_Stdzd_Amt_mean,Avg_Suplr_Sbmtd_Chrg_sum,Avg_Suplr_Mdcr_Alowd_Amt_sum,Avg_Suplr_Mdcr_Pymt_Amt_sum,Avg_Suplr_Mdcr_Stdzd_Amt_sum,Avg_Suplr_Sbmtd_Chrg_median,Avg_Suplr_Mdcr_Alowd_Amt_median,Avg_Suplr_Mdcr_Pymt_Amt_median,Avg_Suplr_Mdcr_Stdzd_Amt_median,Avg_Suplr_Sbmtd_Chrg_std,Avg_Suplr_Mdcr_Alowd_Amt_std,Avg_Suplr_Mdcr_Pymt_Amt_std,Avg_Suplr_Mdcr_Stdzd_Amt_std,Avg_Suplr_Sbmtd_Chrg_min,Avg_Suplr_Mdcr_Alowd_Amt_min,Avg_Suplr_Mdcr_Pymt_Amt_min,Avg_Suplr_Mdcr_Stdzd_Amt_min,Avg_Suplr_Sbmtd_Chrg_max,Avg_Suplr_Mdcr_Alowd_Amt_max,Avg_Suplr_Mdcr_Pymt_Amt_max,Avg_Suplr_Mdcr_Stdzd_Amt_max
0,1003000126,2021,0,129.776563,41.940392,31.813932,34.260562,519.106250,167.761567,127.255730,137.042249,69.168125,29.663750,23.121477,24.416364,156.841976,39.407986,29.041483,32.125772,20.000000,10.210909,8.169091,8.456364,360.770000,98.223158,72.843684,79.753158
1,1003000126,2022,0,209.273634,54.750832,41.764741,52.602407,418.547268,109.501664,83.529481,105.204814,209.273634,54.750832,41.764741,52.602407,227.024317,50.319178,38.478122,50.454058,48.743200,19.169800,14.556600,16.926000,369.804068,90.331864,68.972881,88.278814
2,1003000126,2023,0,152.346111,48.740210,32.699442,42.116461,457.038333,146.220631,98.098326,126.349383,85.053333,21.348889,14.790000,19.462308,142.083999,48.270604,33.771109,41.134005,56.410000,20.396154,11.655385,17.289722,315.575000,104.475588,71.652941,89.597353
3,1003000480,2021,0,272.003846,80.513846,64.407692,84.701538,272.003846,80.513846,64.407692,84.701538,272.003846,80.513846,64.407692,84.701538,NaN,NaN,NaN,NaN,272.003846,80.513846,64.407692,84.701538,272.003846,80.513846,64.407692,84.701538
4,1003000522,2021,0,133.101129,22.805968,18.243952,19.471129,266.202258,45.611935,36.487903,38.942258,133.101129,22.805968,18.243952,19.471129,122.893562,20.030967,16.024522,18.199332,46.202258,8.641935,6.912903,6.602258,220.000000,36.970000,29.575000,32.340000


In [3]:
df_validate_eng.describe()

,npi,Year,target,Avg_Suplr_Sbmtd_Chrg_mean,Avg_Suplr_Mdcr_Alowd_Amt_mean,Avg_Suplr_Mdcr_Pymt_Amt_mean,Avg_Suplr_Mdcr_Stdzd_Amt_mean,Avg_Suplr_Sbmtd_Chrg_sum,Avg_Suplr_Mdcr_Alowd_Amt_sum,Avg_Suplr_Mdcr_Pymt_Amt_sum,Avg_Suplr_Mdcr_Stdzd_Amt_sum,Avg_Suplr_Sbmtd_Chrg_median,Avg_Suplr_Mdcr_Alowd_Amt_median,Avg_Suplr_Mdcr_Pymt_Amt_median,Avg_Suplr_Mdcr_Stdzd_Amt_median,Avg_Suplr_Sbmtd_Chrg_std,Avg_Suplr_Mdcr_Alowd_Amt_std,Avg_Suplr_Mdcr_Pymt_Amt_std,Avg_Suplr_Mdcr_Stdzd_Amt_std,Avg_Suplr_Sbmtd_Chrg_min,Avg_Suplr_Mdcr_Alowd_Amt_min,Avg_Suplr_Mdcr_Pymt_Amt_min,Avg_Suplr_Mdcr_Stdzd_Amt_min,Avg_Suplr_Sbmtd_Chrg_max,Avg_Suplr_Mdcr_Alowd_Amt_max,Avg_Suplr_Mdcr_Pymt_Amt_max,Avg_Suplr_Mdcr_Stdzd_Amt_max
count,8.795310e+05,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,668116.000000,668116.000000,668116.000000,668116.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000,879531.000000
mean,1.499555e+09,2022.002775,0.000235,266.698599,101.789293,78.226587,79.975572,1049.616493,378.473567,289.356365,296.021560,209.021769,81.287966,62.289480,63.670628,261.552555,92.997173,71.835766,73.753444,134.552607,55.106022,42.451403,43.214352,616.807650,226.369378,174.962926,179.182296
std,2.879755e+08,0.816969,0.015339,598.520878,348.298476,274.181022,273.229296,1848.987910,874.229116,681.537007,680.462560,582.810231,341.091256,268.451481,267.521145,436.059058,211.209441,166.375570,166.334912,560.246221,330.938716,260.413106,259.545895,1073.823638,529.619977,416.642499,416.408612
min,1.003000e+09,2021.000000,0.000000,0.011182,0.009000,0.000000,0.000000,0.011182,0.009000,0.000000,0.000000,0.011182,0.009000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002468,0.002468,0.000000,0.000000,0.011182,0.009000,0.000000,0.000000
25%,1.245791e+09,2021.000000,0.000000,66.760555,19.491670,14.206195,15.714411,119.349283,47.201352,35.073730,37.241537,48.228316,8.804981,6.671223,6.533182,62.770778,23.867589,17.858933,20.157002,6.118187,1.422667,0.968725,0.946074,86.310000,34.124545,25.870000,28.196605
50%,1.497910e+09,2022.000000,0.000000,142.623173,50.701297,37.844118,40.370131,482.853590,153.209541,114.870000,117.660000,81.507059,28.352609,20.780321,20.965472,150.231178,48.388445,36.177492,39.858264,25.000000,8.053636,4.927500,5.244000,326.870000,96.986883,72.870000,85.060482
75%,1.750056e+09,2023.000000,0.000000,261.734578,84.620000,64.127892,63.677718,1171.715033,384.547208,290.456772,300.767943,183.315112,61.866390,46.163048,50.907612,270.405395,94.993989,72.481916,71.690789,78.000000,24.300000,17.902685,17.820000,544.812544,244.466571,187.303741,188.998879
max,1.993000e+09,2023.000000,1.000000,38625.000000,15289.440000,11986.920000,11986.920000,178486.725661,121662.723308,94737.962397,93970.493808,38625.000000,15289.440000,11986.920000,11986.920000,39294.651747,10810.989198,8475.802766,8475.871200,36258.694167,15289.440000,11986.920000,11986.920000,77612.100000,27701.767500,21718.189375,21496.420000


In [4]:
# calculate summary stats for non-fraud in df_validate_eng
df_validate_eng_nonfraud = df_validate_eng[df_validate_eng.target==0]
df_validate_eng_nonfraud.describe()

,npi,Year,target,Avg_Suplr_Sbmtd_Chrg_mean,Avg_Suplr_Mdcr_Alowd_Amt_mean,Avg_Suplr_Mdcr_Pymt_Amt_mean,Avg_Suplr_Mdcr_Stdzd_Amt_mean,Avg_Suplr_Sbmtd_Chrg_sum,Avg_Suplr_Mdcr_Alowd_Amt_sum,Avg_Suplr_Mdcr_Pymt_Amt_sum,Avg_Suplr_Mdcr_Stdzd_Amt_sum,Avg_Suplr_Sbmtd_Chrg_median,Avg_Suplr_Mdcr_Alowd_Amt_median,Avg_Suplr_Mdcr_Pymt_Amt_median,Avg_Suplr_Mdcr_Stdzd_Amt_median,Avg_Suplr_Sbmtd_Chrg_std,Avg_Suplr_Mdcr_Alowd_Amt_std,Avg_Suplr_Mdcr_Pymt_Amt_std,Avg_Suplr_Mdcr_Stdzd_Amt_std,Avg_Suplr_Sbmtd_Chrg_min,Avg_Suplr_Mdcr_Alowd_Amt_min,Avg_Suplr_Mdcr_Pymt_Amt_min,Avg_Suplr_Mdcr_Stdzd_Amt_min,Avg_Suplr_Sbmtd_Chrg_max,Avg_Suplr_Mdcr_Alowd_Amt_max,Avg_Suplr_Mdcr_Pymt_Amt_max,Avg_Suplr_Mdcr_Stdzd_Amt_max
count,8.793240e+05,879324.000000,879324.0,879324.000000,879324.000000,879324.000000,879324.000000,879324.000000,879324.000000,879324.000000,879324.000000,879324.000000,879324.000000,879324.000000,879324.000000,667958.000000,667958.000000,667958.000000,667958.000000,879324.000000,879324.000000,879324.000000,879324.000000,879324.000000,879324.000000,879324.000000,879324.000000
mean,1.499546e+09,2022.002849,0.0,266.689980,101.785183,78.223461,79.972606,1049.477003,378.394190,289.295265,295.960318,209.016051,81.285472,62.287546,63.669268,261.536465,92.987844,71.828530,73.746046,134.552363,55.107639,42.452703,43.215656,616.759213,226.346643,174.945363,179.164377
std,2.879730e+08,0.816969,0.0,598.540943,348.327828,274.204357,273.252454,1848.776826,874.023387,681.379246,680.303893,582.829253,341.119754,268.474108,267.543953,436.047760,211.209957,166.376043,166.334066,560.271209,330.971643,260.439157,259.571561,1073.793857,529.619604,416.642597,416.405762
min,1.003000e+09,2021.000000,0.0,0.011182,0.009000,0.000000,0.000000,0.011182,0.009000,0.000000,0.000000,0.011182,0.009000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002468,0.002468,0.000000,0.000000,0.011182,0.009000,0.000000,0.000000
25%,1.245790e+09,2021.000000,0.0,66.759047,19.491476,14.205995,15.713935,119.340000,47.199824,35.072061,37.240748,48.230769,8.805228,6.671250,6.533182,62.767259,23.865223,17.858780,20.156449,6.119541,1.422692,0.968824,0.946154,86.310000,34.124545,25.870000,28.187661
50%,1.497910e+09,2022.000000,0.0,142.625000,50.702135,37.844511,40.371423,482.829539,153.193659,114.867485,117.660000,81.510000,28.354763,20.781572,20.966551,150.224811,48.387413,36.176257,39.857453,25.000000,8.054286,4.927500,5.244191,326.870000,96.982105,72.867446,85.060231
75%,1.750044e+09,2023.000000,0.0,261.731401,84.620000,64.125848,63.676240,1171.547418,384.512707,290.436539,300.732541,183.312732,61.866000,46.162446,50.907551,270.389953,94.990500,72.479864,71.687858,78.000000,24.300000,17.901526,17.820000,544.777688,244.445493,187.292971,188.989305
max,1.993000e+09,2023.000000,0.0,38625.000000,15289.440000,11986.920000,11986.920000,178486.725661,121662.723308,94737.962397,93970.493808,38625.000000,15289.440000,11986.920000,11986.920000,39294.651747,10810.989198,8475.802766,8475.871200,36258.694167,15289.440000,11986.920000,11986.920000,77612.100000,27701.767500,21718.189375,21496.420000


In [5]:
# calculate summary stats for fraud in df_validate_eng
df_validate_eng_fraud = df_validate_eng[df_validate_eng.target==1]
df_validate_eng_fraud.describe()

,npi,Year,target,Avg_Suplr_Sbmtd_Chrg_mean,Avg_Suplr_Mdcr_Alowd_Amt_mean,Avg_Suplr_Mdcr_Pymt_Amt_mean,Avg_Suplr_Mdcr_Stdzd_Amt_mean,Avg_Suplr_Sbmtd_Chrg_sum,Avg_Suplr_Mdcr_Alowd_Amt_sum,Avg_Suplr_Mdcr_Pymt_Amt_sum,Avg_Suplr_Mdcr_Stdzd_Amt_sum,Avg_Suplr_Sbmtd_Chrg_median,Avg_Suplr_Mdcr_Alowd_Amt_median,Avg_Suplr_Mdcr_Pymt_Amt_median,Avg_Suplr_Mdcr_Stdzd_Amt_median,Avg_Suplr_Sbmtd_Chrg_std,Avg_Suplr_Mdcr_Alowd_Amt_std,Avg_Suplr_Mdcr_Pymt_Amt_std,Avg_Suplr_Mdcr_Stdzd_Amt_std,Avg_Suplr_Sbmtd_Chrg_min,Avg_Suplr_Mdcr_Alowd_Amt_min,Avg_Suplr_Mdcr_Pymt_Amt_min,Avg_Suplr_Mdcr_Stdzd_Amt_min,Avg_Suplr_Sbmtd_Chrg_max,Avg_Suplr_Mdcr_Alowd_Amt_max,Avg_Suplr_Mdcr_Pymt_Amt_max,Avg_Suplr_Mdcr_Stdzd_Amt_max
count,2.070000e+02,207.000000,207.0,207.000000,207.000000,207.000000,207.000000,207.000000,207.000000,207.000000,207.000000,207.000000,207.000000,207.000000,207.000000,158.000000,158.000000,158.000000,158.000000,207.000000,207.000000,207.000000,207.000000,207.000000,207.000000,207.000000,207.000000
mean,1.539492e+09,2021.690821,1.0,303.312177,119.248237,91.507086,92.577506,1642.161011,715.662665,548.904335,556.175382,233.313544,91.882504,70.501346,69.449047,329.574769,132.433751,102.425235,105.030145,135.590024,48.239861,36.927033,37.672405,822.562395,322.946186,249.568284,255.300387
std,2.963104e+08,0.757527,0.0,506.058428,185.239850,144.013384,144.345814,2532.632828,1479.514731,1147.536090,1148.421049,496.063320,183.869237,143.141696,140.675175,478.140983,205.913095,162.005141,167.503359,442.587490,130.272677,100.976935,103.862124,1178.624720,523.614192,410.472816,422.548180
min,1.033111e+09,2021.000000,1.0,0.180408,0.140000,0.109755,0.109755,0.180408,0.140000,0.109755,0.109755,0.180408,0.140000,0.109755,0.109755,2.861915,1.274729,1.019209,1.453711,0.133000,0.033980,0.018358,0.018358,0.180408,0.140000,0.109755,0.109755
25%,1.285901e+09,2021.000000,1.0,71.889507,20.279895,15.269165,16.245867,141.586040,59.690000,42.771617,47.612573,38.212608,8.657186,6.258816,6.489489,79.271366,27.400415,19.293615,23.688469,1.403014,0.514786,0.405530,0.374785,106.250000,45.624068,34.250221,34.872549
50%,1.558407e+09,2022.000000,1.0,136.010000,46.700460,35.370951,36.768953,553.660375,191.146471,138.583710,133.195411,73.740769,23.280000,16.695000,16.366119,182.841573,54.986203,41.886640,40.997175,18.090000,4.080000,2.975811,2.763333,367.288136,115.330000,90.257273,85.990000
75%,1.801478e+09,2022.000000,1.0,304.442266,99.074358,75.133229,89.862812,2091.359042,730.671828,546.616339,535.378644,197.911320,71.570040,52.581917,51.452500,306.318658,132.324147,101.753498,97.945864,77.588056,29.340000,21.202051,19.317000,789.562000,268.458452,207.780000,204.497693
max,1.992736e+09,2023.000000,1.0,3317.739231,1060.380000,831.330000,857.867500,14052.424339,10165.239610,7829.982824,7994.944681,3317.739231,1060.380000,831.330000,857.867500,3080.997689,1106.998018,874.145154,870.072603,3317.739231,1060.380000,831.330000,857.867500,5457.342222,3026.970769,2359.514615,2696.019231


# Engineer features on other datasets to adjoin to this one

Features must be aggregated by NPI and class label. The DMEPOS by referring provider dataset includes totals of number of suppliers, number of supplier HCPCS, number of supplier beneficiaries, and number of supplier claims by referring provider NPI but ignores whether the supplies were rented which is a feature that is potentially informative. As such, we elect to perform additional feature engineering on the DMEPOS by referring provider and service dataset and use the provider-level dataset for patient demographics not included within the more granular-level data.

## Grouping by npi, Suplr_Rentl_Ind, and target, calculate count of HCPCS_CD and mean, sum, median, std, min, and max for Tot_Suplrs, Tot_Suplr_Benes, Tot_Suplr_Clms, and Tot_Suplr_Srvcs.

In [6]:
# first convert Suplr_Rentl_Ind to numeric type
rentl_map = {'N':0,'Y':1}
df_validate['Suplr_Rentl_Ind'] = df_validate['Suplr_Rentl_Ind'].map(rentl_map)


In [12]:
# compute total HCPCS cds for each provider by rental indicator
df_new_fts = df_validate[['npi','HCPCS_CD','Suplr_Rentl_Ind','Year']].groupby(['npi','Suplr_Rentl_Ind','Year']).nunique().drop(['npi','Suplr_Rentl_Ind','Year'],axis=1).reset_index()
df_new_fts = df_new_fts.rename(columns={'HCPCS_CD':'Tot_Suplr_HCPCS_Cds'}) # Rename to Tot_Suplr_HCPCS_Cds to keep with CMS convention
# transform into rented and not rented totals
df_new_fts = pd.pivot_table(df_new_fts, values="Tot_Suplr_HCPCS_Cds", index=["npi","Year"], columns=["Suplr_Rentl_Ind"]).reset_index()
df_new_fts = df_new_fts.rename(columns={0:'Tot_Suplr_Nonrntl_HCPCS_Cds',1:'Tot_Suplr_Rentl_HCPCS_Cds'})
df_new_fts = df_new_fts.fillna(0) # impute 0 for NaN
df_new_fts = df_new_fts.rename_axis(None, axis=1) # get rid of index name
df_new_fts.head()

,npi,Year,Tot_Suplr_Nonrntl_HCPCS_Cds,Tot_Suplr_Rentl_HCPCS_Cds
0,1003000126,2021,0.0,4.0
1,1003000126,2022,0.0,2.0
2,1003000126,2023,0.0,3.0
3,1003000480,2021,0.0,1.0
4,1003000522,2021,1.0,1.0


In [13]:
# add engineered features
df_new_fts[['Tot_Suplrs_mean','Tot_Suplr_Benes_mean','Tot_Suplr_Clms_mean','Tot_Suplr_Srvcs_mean']] = df_validate[['npi','Tot_Suplrs','Tot_Suplr_Benes','Tot_Suplr_Clms','Tot_Suplr_Srvcs','Year']].groupby(['npi','Year']).mean().reset_index(drop=True)
df_new_fts[['Tot_Suplrs_sum','Tot_Suplr_Benes_sum','Tot_Suplr_Clms_sum','Tot_Suplr_Srvcs_sum']] = df_validate[['npi','Tot_Suplrs','Tot_Suplr_Benes','Tot_Suplr_Clms','Tot_Suplr_Srvcs','Year']].groupby(['npi','Year']).sum().reset_index(drop=True)
df_new_fts[['Tot_Suplrs_median','Tot_Suplr_Benes_median','Tot_Suplr_Clms_median','Tot_Suplr_Srvcs_median']] = df_validate[['npi','Tot_Suplrs','Tot_Suplr_Benes','Tot_Suplr_Clms','Tot_Suplr_Srvcs','Year']].groupby(['npi','Year']).median().reset_index(drop=True)
df_new_fts[['Tot_Suplrs_std','Tot_Suplr_Benes_std','Tot_Suplr_Clms_std','Tot_Suplr_Srvcs_std']] = df_validate[['npi','Tot_Suplrs','Tot_Suplr_Benes','Tot_Suplr_Clms','Tot_Suplr_Srvcs','Year']].groupby(['npi','Year']).std().reset_index(drop=True)
df_new_fts[['Tot_Suplrs_min','Tot_Suplr_Benes_min','Tot_Suplr_Clms_min','Tot_Suplr_Srvcs_min']] = df_validate[['npi','Tot_Suplrs','Tot_Suplr_Benes','Tot_Suplr_Clms','Tot_Suplr_Srvcs','Year']].groupby(['npi','Year']).min().reset_index(drop=True)
df_new_fts[['Tot_Suplrs_max','Tot_Suplr_Benes_max','Tot_Suplr_Clms_max','Tot_Suplr_Srvcs_max']] = df_validate[['npi','Tot_Suplrs','Tot_Suplr_Benes','Tot_Suplr_Clms','Tot_Suplr_Srvcs','Year']].groupby(['npi','Year']).max().reset_index(drop=True)
df_new_fts.head()

,npi,Year,Tot_Suplr_Nonrntl_HCPCS_Cds,Tot_Suplr_Rentl_HCPCS_Cds,Tot_Suplrs_mean,Tot_Suplr_Benes_mean,Tot_Suplr_Clms_mean,Tot_Suplr_Srvcs_mean,Tot_Suplrs_sum,Tot_Suplr_Benes_sum,Tot_Suplr_Clms_sum,Tot_Suplr_Srvcs_sum,Tot_Suplrs_median,Tot_Suplr_Benes_median,Tot_Suplr_Clms_median,Tot_Suplr_Srvcs_median,Tot_Suplrs_std,Tot_Suplr_Benes_std,Tot_Suplr_Clms_std,Tot_Suplr_Srvcs_std,Tot_Suplrs_min,Tot_Suplr_Benes_min,Tot_Suplr_Clms_min,Tot_Suplr_Srvcs_min,Tot_Suplrs_max,Tot_Suplr_Benes_max,Tot_Suplr_Clms_max,Tot_Suplr_Srvcs_max
0,1003000126,2021,0.0,4.0,3.250000,5.0,14.250000,14.250000,13,20.0,57,57,3.0,5.0,13.5,13.5,2.629956,0.000000,3.947573,3.947573,1,5.0,11,11,6,5.0,19,19
1,1003000126,2022,0.0,2.0,4.500000,8.0,54.500000,54.500000,9,16.0,109,109,4.5,8.0,54.5,54.5,0.707107,4.242641,6.363961,6.363961,4,5.0,50,50,5,11.0,59,59
2,1003000126,2023,0.0,3.0,2.666667,5.0,27.666667,27.666667,8,15.0,83,83,3.0,5.0,34.0,34.0,1.527525,0.000000,12.741010,12.741010,1,5.0,13,13,4,5.0,36,36
3,1003000480,2021,0.0,1.0,4.000000,5.0,11.000000,13.000000,4,5.0,11,13,4.0,5.0,11.0,13.0,NaN,NaN,NaN,NaN,4,5.0,11,13,4,5.0,11,13
4,1003000522,2021,1.0,1.0,2.000000,5.0,12.000000,21.500000,4,10.0,24,43,2.0,5.0,12.0,21.5,1.414214,0.000000,0.000000,13.435029,1,5.0,12,12,3,5.0,12,31


In [14]:
# join the other engineered features on npi and Year
df_eng = df_validate_eng.merge(df_new_fts, on=["npi","Year"])
df_eng.head()
# df_new_fts.shape

,npi,Year,target,Avg_Suplr_Sbmtd_Chrg_mean,Avg_Suplr_Mdcr_Alowd_Amt_mean,Avg_Suplr_Mdcr_Pymt_Amt_mean,Avg_Suplr_Mdcr_Stdzd_Amt_mean,Avg_Suplr_Sbmtd_Chrg_sum,Avg_Suplr_Mdcr_Alowd_Amt_sum,Avg_Suplr_Mdcr_Pymt_Amt_sum,Avg_Suplr_Mdcr_Stdzd_Amt_sum,Avg_Suplr_Sbmtd_Chrg_median,Avg_Suplr_Mdcr_Alowd_Amt_median,Avg_Suplr_Mdcr_Pymt_Amt_median,Avg_Suplr_Mdcr_Stdzd_Amt_median,Avg_Suplr_Sbmtd_Chrg_std,Avg_Suplr_Mdcr_Alowd_Amt_std,Avg_Suplr_Mdcr_Pymt_Amt_std,Avg_Suplr_Mdcr_Stdzd_Amt_std,Avg_Suplr_Sbmtd_Chrg_min,Avg_Suplr_Mdcr_Alowd_Amt_min,Avg_Suplr_Mdcr_Pymt_Amt_min,Avg_Suplr_Mdcr_Stdzd_Amt_min,Avg_Suplr_Sbmtd_Chrg_max,Avg_Suplr_Mdcr_Alowd_Amt_max,Avg_Suplr_Mdcr_Pymt_Amt_max,Avg_Suplr_Mdcr_Stdzd_Amt_max,Tot_Suplr_Nonrntl_HCPCS_Cds,Tot_Suplr_Rentl_HCPCS_Cds,Tot_Suplrs_mean,Tot_Suplr_Benes_mean,Tot_Suplr_Clms_mean,Tot_Suplr_Srvcs_mean,Tot_Suplrs_sum,Tot_Suplr_Benes_sum,Tot_Suplr_Clms_sum,Tot_Suplr_Srvcs_sum,Tot_Suplrs_median,Tot_Suplr_Benes_median,Tot_Suplr_Clms_median,Tot_Suplr_Srvcs_median,Tot_Suplrs_std,Tot_Suplr_Benes_std,Tot_Suplr_Clms_std,Tot_Suplr_Srvcs_std,Tot_Suplrs_min,Tot_Suplr_Benes_min,Tot_Suplr_Clms_min,Tot_Suplr_Srvcs_min,Tot_Suplrs_max,Tot_Suplr_Benes_max,Tot_Suplr_Clms_max,Tot_Suplr_Srvcs_max
0,1003000126,2021,0,129.776563,41.940392,31.813932,34.260562,519.106250,167.761567,127.255730,137.042249,69.168125,29.663750,23.121477,24.416364,156.841976,39.407986,29.041483,32.125772,20.000000,10.210909,8.169091,8.456364,360.770000,98.223158,72.843684,79.753158,0.0,4.0,3.250000,5.0,14.250000,14.250000,13,20.0,57,57,3.0,5.0,13.5,13.5,2.629956,0.000000,3.947573,3.947573,1,5.0,11,11,6,5.0,19,19
1,1003000126,2022,0,209.273634,54.750832,41.764741,52.602407,418.547268,109.501664,83.529481,105.204814,209.273634,54.750832,41.764741,52.602407,227.024317,50.319178,38.478122,50.454058,48.743200,19.169800,14.556600,16.926000,369.804068,90.331864,68.972881,88.278814,0.0,2.0,4.500000,8.0,54.500000,54.500000,9,16.0,109,109,4.5,8.0,54.5,54.5,0.707107,4.242641,6.363961,6.363961,4,5.0,50,50,5,11.0,59,59
2,1003000126,2023,0,152.346111,48.740210,32.699442,42.116461,457.038333,146.220631,98.098326,126.349383,85.053333,21.348889,14.790000,19.462308,142.083999,48.270604,33.771109,41.134005,56.410000,20.396154,11.655385,17.289722,315.575000,104.475588,71.652941,89.597353,0.0,3.0,2.666667,5.0,27.666667,27.666667,8,15.0,83,83,3.0,5.0,34.0,34.0,1.527525,0.000000,12.741010,12.741010,1,5.0,13,13,4,5.0,36,36
3,1003000480,2021,0,272.003846,80.513846,64.407692,84.701538,272.003846,80.513846,64.407692,84.701538,272.003846,80.513846,64.407692,84.701538,NaN,NaN,NaN,NaN,272.003846,80.513846,64.407692,84.701538,272.003846,80.513846,64.407692,84.701538,0.0,1.0,4.000000,5.0,11.000000,13.000000,4,5.0,11,13,4.0,5.0,11.0,13.0,NaN,NaN,NaN,NaN,4,5.0,11,13,4,5.0,11,13
4,1003000522,2021,0,133.101129,22.805968,18.243952,19.471129,266.202258,45.611935,36.487903,38.942258,133.101129,22.805968,18.243952,19.471129,122.893562,20.030967,16.024522,18.199332,46.202258,8.641935,6.912903,6.602258,220.000000,36.970000,29.575000,32.340000,1.0,1.0,2.000000,5.0,12.000000,21.500000,4,10.0,24,43,2.0,5.0,12.0,21.5,1.414214,0.000000,0.000000,13.435029,1,5.0,12,12,3,5.0,12,31


In [15]:
df_eng.shape

(879531, 53)

## Pull in the provider-level data and engineer features on beneficiary demographic data

In [17]:
rfrr = pd.read_csv('/dsa/groups/casestudycf25/team02/DMEPOS_rfrr_clean_labeled.csv',dtype={'Rfrg_Prvdr_State_FIPS':str,'Rfrg_Prvdr_Zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str
rfrr.head()

,npi,Rfrg_Prvdr_Last_Name_Org,Rfrg_Prvdr_First_Name,Rfrg_Prvdr_MI,Rfrg_Prvdr_Crdntls,Rfrg_Prvdr_Ent_Cd,Rfrg_Prvdr_St1,Rfrg_Prvdr_St2,Rfrg_Prvdr_City,Rfrg_Prvdr_State_Abrvtn,Rfrg_Prvdr_State_FIPS,Rfrg_Prvdr_Zip5,Rfrg_Prvdr_RUCA,Rfrg_Prvdr_RUCA_Desc,Rfrg_Prvdr_Cntry,Rfrg_Prvdr_Spclty_Desc,Rfrg_Prvdr_Spclty_Srce,Tot_Suplrs,Tot_Suplr_HCPCS_Cds,Tot_Suplr_Benes,Tot_Suplr_Clms,Tot_Suplr_Srvcs,Suplr_Sbmtd_Chrgs,Suplr_Mdcr_Alowd_Amt,Suplr_Mdcr_Pymt_Amt,Suplr_Mdcr_Stdzd_Pymt_Amt,DME_Sprsn_Ind,DME_Tot_Suplrs,DME_Tot_Suplr_HCPCS_Cds,DME_Tot_Suplr_Benes,DME_Tot_Suplr_Clms,DME_Tot_Suplr_Srvcs,DME_Suplr_Sbmtd_Chrgs,DME_Suplr_Mdcr_Alowd_Amt,DME_Suplr_Mdcr_Pymt_Amt,DME_Suplr_Mdcr_Stdzd_Pymt_Amt,POS_Sprsn_Ind,POS_Tot_Suplrs,POS_Tot_Suplr_HCPCS_Cds,POS_Tot_Suplr_Benes,POS_Tot_Suplr_Clms,POS_Tot_Suplr_Srvcs,POS_Suplr_Sbmtd_Chrgs,POS_Suplr_Mdcr_Alowd_Amt,POS_Suplr_Mdcr_Pymt_Amt,POS_Suplr_Mdcr_Stdzd_Pymt_Amt,Drug_Sprsn_Ind,Drug_Tot_Suplrs,Drug_Tot_Suplr_HCPCS_Cds,Drug_Tot_Suplr_Benes,Drug_Tot_Suplr_Clms,Drug_Tot_Suplr_Srvcs,Drug_Suplr_Sbmtd_Chrgs,Drug_Suplr_Mdcr_Alowd_Amt,Drug_Suplr_Mdcr_Pymt_Amt,Drug_Suplr_Mdcr_Stdzd_Pymt_Amt,Bene_Avg_Age,Bene_Age_LT_65_Cnt,Bene_Age_65_74_Cnt,Bene_Age_75_84_Cnt,Bene_Age_GT_84_Cnt,Bene_Feml_Cnt,Bene_Male_Cnt,Bene_Race_Wht_Cnt,Bene_Race_Black_Cnt,Bene_Race_Api_Cnt,Bene_Race_Hspnc_Cnt,Bene_Race_Natind_Cnt,Bene_Race_Othr_Cnt,Bene_Ndual_Cnt,Bene_Dual_Cnt,Bene_CC_BH_ADHD_OthCD_V1_Pct,Bene_CC_BH_Alcohol_Drug_V1_Pct,Bene_CC_BH_Tobacco_V1_Pct,Bene_CC_BH_Alz_NonAlzdem_V2_Pct,Bene_CC_BH_Anxiety_V1_Pct,Bene_CC_BH_Bipolar_V1_Pct,Bene_CC_BH_Mood_V2_Pct,Bene_CC_BH_Depress_V1_Pct,Bene_CC_BH_PD_V1_Pct,Bene_CC_BH_PTSD_V1_Pct,Bene_CC_BH_Schizo_OthPsy_V1_Pct,Bene_CC_PH_Asthma_V2_Pct,Bene_CC_PH_Afib_V2_Pct,Bene_CC_PH_Cancer6_V2_Pct,Bene_CC_PH_CKD_V2_Pct,Bene_CC_PH_COPD_V2_Pct,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre,Year,target
0,1972596757,Naushad,Abdul,N,md,I,2865 James Blvd,NaN,Poplar Bluff,MO,29,63901,4.0,Micropolitan area core: primary flow within an...,US,Pain Management,Claim-Specialty,6,14,5.0,22,27,13371.30,10217.71,7784.70,7384.51,NaN,2.0,3.0,5.0,11.0,12.0,1380.61,862.55,658.97,436.15,NaN,6.0,11.0,0.0,11.0,15.0,11990.69,9355.16,7125.73,6948.36,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64.125000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,1.255375,2021,1
1,1265697478,Balger,Abigail,NaN,md,I,805 E Geneva Dr,NaN,Dewitt,MI,26,48820,1.0,Metropolitan area core: primary flow within an...,US,Emergency Medicine,NPPES-Specialty,1,8,34.0,34,99,75655.00,53841.66,40491.75,38453.43,NaN,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,NaN,1.0,8.0,34.0,34.0,99.0,75655.00,53841.66,40491.75,38453.43,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,68.352941,0.0,16.0,13.0,0.0,20.0,14.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.441176,0.0,0.411765,0.323529,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.411765,0.000000,0.735294,0.794118,0.000000,0.0,0.0,0.529412,0.0,1.433834,2021,1
2,1922096221,Alperovich,Alexander,NaN,md,I,1340 Union University Dr,NaN,Jackson,TN,47,38305,1.0,Metropolitan area core: primary flow within an...,US,Cardiology,Claim-Specialty,9,24,33.0,137,318,171131.05,78376.77,62002.29,59621.86,NaN,9.0,24.0,33.0,137.0,318.0,171131.05,78376.77,62002.29,59621.86,NaN,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,76.636364,0.0,12.0,14.0,0.0,16.0,17.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,0.0,0.333333,0.000000,0.0,0.0,0.0,0.0,0.393939,0.0,0.363636,0.393939,0.545455,0.666667,0.878788,0.969697,0.727273,0.0,0.0,0.666667,0.0,1.858711,2021,1
3,1922096221,Alperovich,Alexander,NaN,md,I,1340

### Initially tried grouping by NPI. Cells maintained as Raw NB Convert for reference only.

We find 498,197 records when grouping the referring provider level dataset on NPI and fraud class label as opposed to 398,034 in the referring provider and service dataset when grouping by the same features. We elect to inner join the two datasets for analysis.

### Group by NPI-Year

In [18]:
rfrr_cols = rfrr.columns[56:-2] # beneficiary demographic columns

# importand cols for joining added to index
idx = pd.Index(['npi'])
idx = idx.append(rfrr_cols)
idx = idx.append(pd.Index(['Year']))

rfrr_fts = rfrr[idx]

In [19]:
# join rfrr and df_eng
df_eng = df_eng.merge(rfrr_fts, on=['npi','Year'])
df_eng.head()

,npi,Year,target,Avg_Suplr_Sbmtd_Chrg_mean,Avg_Suplr_Mdcr_Alowd_Amt_mean,Avg_Suplr_Mdcr_Pymt_Amt_mean,Avg_Suplr_Mdcr_Stdzd_Amt_mean,Avg_Suplr_Sbmtd_Chrg_sum,Avg_Suplr_Mdcr_Alowd_Amt_sum,Avg_Suplr_Mdcr_Pymt_Amt_sum,Avg_Suplr_Mdcr_Stdzd_Amt_sum,Avg_Suplr_Sbmtd_Chrg_median,Avg_Suplr_Mdcr_Alowd_Amt_median,Avg_Suplr_Mdcr_Pymt_Amt_median,Avg_Suplr_Mdcr_Stdzd_Amt_median,Avg_Suplr_Sbmtd_Chrg_std,Avg_Suplr_Mdcr_Alowd_Amt_std,Avg_Suplr_Mdcr_Pymt_Amt_std,Avg_Suplr_Mdcr_Stdzd_Amt_std,Avg_Suplr_Sbmtd_Chrg_min,Avg_Suplr_Mdcr_Alowd_Amt_min,Avg_Suplr_Mdcr_Pymt_Amt_min,Avg_Suplr_Mdcr_Stdzd_Amt_min,Avg_Suplr_Sbmtd_Chrg_max,Avg_Suplr_Mdcr_Alowd_Amt_max,Avg_Suplr_Mdcr_Pymt_Amt_max,Avg_Suplr_Mdcr_Stdzd_Amt_max,Tot_Suplr_Nonrntl_HCPCS_Cds,Tot_Suplr_Rentl_HCPCS_Cds,Tot_Suplrs_mean,Tot_Suplr_Benes_mean,Tot_Suplr_Clms_mean,Tot_Suplr_Srvcs_mean,Tot_Suplrs_sum,Tot_Suplr_Benes_sum,Tot_Suplr_Clms_sum,Tot_Suplr_Srvcs_sum,Tot_Suplrs_median,Tot_Suplr_Benes_median,Tot_Suplr_Clms_median,Tot_Suplr_Srvcs_median,Tot_Suplrs_std,Tot_Suplr_Benes_std,Tot_Suplr_Clms_std,Tot_Suplr_Srvcs_std,Tot_Suplrs_min,Tot_Suplr_Benes_min,Tot_Suplr_Clms_min,Tot_Suplr_Srvcs_min,Tot_Suplrs_max,Tot_Suplr_Benes_max,Tot_Suplr_Clms_max,Tot_Suplr_Srvcs_max,Bene_Avg_Age,Bene_Age_LT_65_Cnt,Bene_Age_65_74_Cnt,Bene_Age_75_84_Cnt,Bene_Age_GT_84_Cnt,Bene_Feml_Cnt,Bene_Male_Cnt,Bene_Race_Wht_Cnt,Bene_Race_Black_Cnt,Bene_Race_Api_Cnt,Bene_Race_Hspnc_Cnt,Bene_Race_Natind_Cnt,Bene_Race_Othr_Cnt,Bene_Ndual_Cnt,Bene_Dual_Cnt,Bene_CC_BH_ADHD_OthCD_V1_Pct,Bene_CC_BH_Alcohol_Drug_V1_Pct,Bene_CC_BH_Tobacco_V1_Pct,Bene_CC_BH_Alz_NonAlzdem_V2_Pct,Bene_CC_BH_Anxiety_V1_Pct,Bene_CC_BH_Bipolar_V1_Pct,Bene_CC_BH_Mood_V2_Pct,Bene_CC_BH_Depress_V1_Pct,Bene_CC_BH_PD_V1_Pct,Bene_CC_BH_PTSD_V1_Pct,Bene_CC_BH_Schizo_OthPsy_V1_Pct,Bene_CC_PH_Asthma_V2_Pct,Bene_CC_PH_Afib_V2_Pct,Bene_CC_PH_Cancer6_V2_Pct,Bene_CC_PH_CKD_V2_Pct,Bene_CC_PH_COPD_V2_Pct,Bene_CC_PH_Diabetes_V2_Pct,Bene_CC_PH_HF_NonIHD_V2_Pct,Bene_CC_PH_Hyperlipidemia_V2_Pct,Bene_CC_PH_Hypertension_V2_Pct,Bene_CC_PH_IschemicHeart_V2_Pct,Bene_CC_PH_Osteoporosis_V2_Pct,Bene_CC_PH_Parkinson_V2_Pct,Bene_CC_PH_Arthritis_V2_Pct,Bene_CC_PH_Stroke_TIA_V2_Pct,Bene_Avg_Risk_Scre
0,1003000126,2021,0,129.776563,41.940392,31.813932,34.260562,519.106250,167.761567,127.255730,137.042249,69.168125,29.663750,23.121477,24.416364,156.841976,39.407986,29.041483,32.125772,20.000000,10.210909,8.169091,8.456364,360.770000,98.223158,72.843684,79.753158,0.0,4.0,3.250000,5.0,14.250000,14.250000,13,20.0,57,57,3.0,5.0,13.5,13.5,2.629956,0.000000,3.947573,3.947573,1,5.0,11,11,6,5.0,19,19,79.750000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.000000,0.0,0.0,0.0,0.0,1.942167
1,1003000126,2022,0,209.273634,54.750832,41.764741,52.602407,418.547268,109.501664,83.529481,105.204814,209.273634,54.750832,41.764741,52.602407,227.024317,50.319178,38.478122,50.454058,48.743200,19.169800,14.556600,16.926000,369.804068,90.331864,68.972881,88.278814,0.0,2.0,4.500000,8.0,54.500000,54.500000,9,16.0,109,109,4.5,8.0,54.5,54.5,0.707107,4.242641,6.363961,6.363961,4,5.0,50,50,5,11.0,59,59,76.388889,0.0,0.0,0.0,0.0,0.0,0.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.833333,0.944444,0.777778,0.0,0.0,0.0,0.0,2.987489
2,1003000126,2023,0,152.346111,48.740210,32.699442,42.116461,457.038333,146.220631,98.098326,126.349383,85.053333,21.348889,14.790000,19.462308,142.083999,48.270604,33.771109,41.134005,56.410000,20.396154,11.655385,17.289722,315.575000,104.475588,71.652941,89.597353,0.0,3.0,2.666667,5.0,27.666667,27.666667,8,15.0,83,83,3.0,5.0,34.0,34.0,1.527525,0.000000,12.741010,12.741010,1,5.0,13,13,4,5.0,36,36,72.923077,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000,0.000000,0.0,0.0,0.0,0.0,3.500804
3,1003000480,2021,0,272.003846,80.513846,64.407692,84

In [20]:
df_eng.shape

(879531, 94)

In [21]:
# save df_eng as a csv
df_eng.to_csv('/dsa/groups/casestudycf25/team02/DMEPOS_Amount_Stats_labeled.csv', index=False)

PermissionError: [Errno 13] Permission denied: '/dsa/groups/casestudycf25/team02/DMEPOS_Amount_Stats_labeled.csv'